In [1]:
#@title Imports
import html
import ipywidgets
import importlib
import os
from pathlib import Path
import re
import subprocess
import sys

### Analyst v4
A notebook interface for asking questions about certain security conference proceedings using a prepared knowledge base. SQL queries can alternatively be used to search for researchers, talks, or tools. Answers to free-form questions are given from the pre-processed topic summaries. There are three dozen summaries covering topics across the conference sources. There are also summaries for each conference and for authors with more than one presentation in the data. Populate an openAI key as a secret (in colab) or in a .env file if running locally (example file provided) to make LLM calls; SQL queries need no key.

(source repo: (https://github.com/opendr-io/analyst)

The next cell is needed only when using this notebook in Google Colab. Local Jupyter users can skip it.

In [2]:
#@title Colab Setup
# Optional Google Colab bootstrap.
# Local Jupyter users can skip this cell.
try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ANALYST_REPO_URL = os.environ.get("ANALYST_REPO_URL", "https://github.com/opendr-io/analyst.git")
ANALYST_ROOT = Path(os.environ.get("ANALYST_ROOT", "/content/analyst" if IN_COLAB else Path.cwd()))
ANALYST_REPO_REF = os.environ.get("ANALYST_REPO_REF", "")  # Optional branch or tag, useful for forks/testing.
COLAB_QA_MODEL = os.environ.get("COLAB_QA_MODEL", "")  # Optional, for accounts without the repo default QA model.

if IN_COLAB:
    if COLAB_QA_MODEL and not os.environ.get("LLM_MODEL_QA"):
        os.environ["LLM_MODEL_QA"] = COLAB_QA_MODEL

    if not (ANALYST_ROOT / "the_analyst.py").exists():
        if not ANALYST_REPO_URL:
            raise RuntimeError(
                "Set ANALYST_REPO_URL to the public repo URL before running this Colab setup cell."
            )
        clone_cmd = ["git", "clone", "--depth", "1"]
        if ANALYST_REPO_REF:
            clone_cmd.extend(["--branch", ANALYST_REPO_REF])
        clone_cmd.extend([ANALYST_REPO_URL, str(ANALYST_ROOT)])
        subprocess.run(clone_cmd, check=True)

    requirements_path = ANALYST_ROOT / "requirements-colab.txt"
    if not requirements_path.exists():
        requirements_path = ANALYST_ROOT / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)], check=True)
    os.chdir(ANALYST_ROOT)

    try:
        from google.colab import output  # type: ignore[import-not-found]
        output.enable_custom_widget_manager()
    except Exception:
        pass

    print(f"Analyst root: {ANALYST_ROOT}")
else:
    print("Not running in Colab; continuing with the local environment.")


Not running in Colab; continuing with the local environment.


Optional: specify a model for questions. Leave it blank to use `.env` locally or `config/llm.ini` as the repo fallback.


In [3]:
#@title Optional LLM Configuration
# Leave blank to use .env locally or config/llm.ini as the repo fallback.
# If not set, the default model will be used. 
LLM_MODEL_QA = ""  #@param {type:"string"}

if LLM_MODEL_QA.strip():
    os.environ["ANALYST_NOTEBOOK_LLM_MODEL_QA"] = LLM_MODEL_QA.strip()
else:
    os.environ.pop("ANALYST_NOTEBOOK_LLM_MODEL_QA", None)

print("Notebook QA model override=", os.environ.get("ANALYST_NOTEBOOK_LLM_MODEL_QA", "none"))

Notebook QA model override= none


In [4]:
#@title Initialize Analyst

def find_analyst_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "the_analyst.py").exists():
            return path
    raise RuntimeError("Could not find the Analyst repo root. Start Jupyter from the repo or set the path manually.")


def load_colab_secrets() -> list[str]:
    """Load optional provider API keys from Colab Secrets when available.

    In Colab, secrets intentionally override existing environment values so a
    corrected secret can replace a stale or invalid key after rerunning this cell.
    """
    if "google.colab" not in sys.modules:
        return []
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
    except Exception:
        return []

    loaded: list[str] = []
    for name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            loaded.append(name)
    return loaded

ROOT = find_analyst_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
# Let the repo .env replace stale keys already present in the notebook kernel.
load_dotenv(ROOT / ".env", override=True)
# Reapply explicit notebook QA model override after .env is loaded.
if os.environ.get("ANALYST_NOTEBOOK_LLM_MODEL_QA"):
    os.environ["LLM_MODEL_QA"] = os.environ["ANALYST_NOTEBOOK_LLM_MODEL_QA"]
elif "ANALYST_NOTEBOOK_LLM_MODEL_QA" in os.environ:
    os.environ.pop("LLM_MODEL_QA", None)
loaded_colab_secrets = load_colab_secrets()
if loaded_colab_secrets:
    print("Loaded Colab secret(s): " + ", ".join(loaded_colab_secrets))

import the_analyst as analyst
from knowledge_indexing import knowledge_index as ki
from knowledge_agenting import topic_summarizer as ts
ts = importlib.reload(ts)

print(f"Analyst root: {ROOT}")


Analyst root: \\vmware-host\Shared Folders\SHARED\GitHub\analyst


In [5]:
#@title Helper Functions
# Core functions

def set_openai_key(value: str | None = None) -> str:
    """Set or replace OPENAI_API_KEY for the current notebook runtime."""
    if value is None:
        import getpass
        value = getpass.getpass("OPENAI_API_KEY: ")
    value = value.strip()
    if not value:
        return "OPENAI_API_KEY was not changed."
    os.environ["OPENAI_API_KEY"] = value
    return "OPENAI_API_KEY updated for this notebook runtime."

def clear_openai_key() -> str:
    """Remove OPENAI_API_KEY from the current notebook runtime."""
    os.environ.pop("OPENAI_API_KEY", None)
    return "OPENAI_API_KEY cleared for this notebook runtime."

def set_anthropic_key(value: str | None = None) -> str:
    """Set or replace ANTHROPIC_API_KEY for the current notebook runtime."""
    if value is None:
        import getpass
        value = getpass.getpass("ANTHROPIC_API_KEY: ")
    value = value.strip()
    if not value:
        return "ANTHROPIC_API_KEY was not changed."
    os.environ["ANTHROPIC_API_KEY"] = value
    return "ANTHROPIC_API_KEY updated for this notebook runtime."

def clear_anthropic_key() -> str:
    """Remove ANTHROPIC_API_KEY from the current notebook runtime."""
    os.environ.pop("ANTHROPIC_API_KEY", None)
    return "ANTHROPIC_API_KEY cleared for this notebook runtime."

def capture_analyst_print(fn, *args, **kwargs) -> str:
    """Capture Analyst output that normally goes through ki.safe_print."""
    lines: list[str] = []
    original_safe_print = ki.safe_print

    def notebook_print(value: str = "") -> None:
        lines.append(str(value))

    ki.safe_print = notebook_print
    try:
        fn(*args, **kwargs)
    finally:
        ki.safe_print = original_safe_print
    return "\n".join(lines).strip()


def ask_summaries_only(question: str, max_summaries: int = 3) -> str:
    """Ask using only routed summary evidence, with up to max_summaries loaded."""
    question = question.strip()
    if not question:
        return "Enter a question."
    max_summaries = max(1, int(max_summaries))
    evidence = analyst.tool_answer_from_summaries(question=question, max_topics=max_summaries)
    if evidence.startswith("No summary files found"):
        return evidence
    client = analyst.llm_client.create_client()
    try:
        response = client.messages.create(
            model=analyst.MODEL,
            max_tokens=analyst.QA_MAX_OUTPUT_TOKENS,
            system=analyst.AGENT_SYSTEM,
            messages=[
                {
                    "role": "user",
                    "content": (
                        f"Question: {question}\n\n"
                        "Answer using ONLY the summary evidence below. "
                        "Start with a Sources used line naming every summary file you relied on. "
                        "Then answer directly in natural prose. "
                        "If the summary evidence is thin or does not answer the question, say so plainly.\n\n"
                        f"{evidence}"
                    ),
                }
            ],
        )
    except Exception as exc:
        status_code = getattr(exc, "status_code", None)
        message = str(exc)
        if status_code == 401 or "invalid_api_key" in message or "Incorrect API key" in message:
            provider_name = analyst.llm_settings.get_provider()
            if provider_name in {"anthropic", "claude"}:
                secret_name = "ANTHROPIC_API_KEY"
                setter_name = "set_anthropic_key()"
                provider_label = "Anthropic"
            else:
                secret_name = "OPENAI_API_KEY"
                setter_name = "set_openai_key()"
                provider_label = "OpenAI"
            if "google.colab" in sys.modules:
                return (
                    f"{provider_label} rejected the configured API key. Update the Colab Secret named "
                    f"{secret_name}, then rerun the setup/import cells. You can also run "
                    f"{setter_name} to replace it for this runtime. No-LLM modes still work."
                )
            return (
                f"{provider_label} rejected the configured API key. Update {secret_name} in the repo .env, "
                "rerun the Initialize Analyst and Helper Functions cells, then try again. You can "
                f"also run {setter_name} to replace it for this notebook runtime. No-LLM modes "
                "still work."
            )
        raise
    answer = "\n".join(block.text for block in response.content if hasattr(block, "text")).strip()
    provider_name = analyst.llm_settings.get_provider()
    return f"Model: {provider_name}/{analyst.MODEL}\n\n{answer}"


def query_annotations(query: str, limit: int | None = 10) -> str:
    """Search structured annotations without using the LLM."""
    query = query.strip()
    if not query:
        return "Enter a query."
    return analyst.tool_query_annotations(query=query, limit=limit)


def analyst_status() -> str:
    return analyst.tool_get_status()


def list_summary_topics() -> str:
    """List topics that have classified records and can be summarized."""
    topics = ts.list_topics_for_summary(ki.DB_PATH)
    if not topics:
        return "No topics with classified records found."
    return "\n".join([*topics, f"topics: {len(topics)}"])


def list_topic_summaries() -> str:
    """List existing topic summary Markdown files."""
    if hasattr(ts, "list_existing_summaries"):
        summaries = ts.list_existing_summaries(ROOT / "summaries", "topic")
        if not summaries:
            return "No topic summary files found."
        lines = ["topic\tstatus\trecords\tgenerated_at\tpath"]
        for item in summaries:
            records = "" if item.records is None else str(item.records)
            lines.append(f"{item.name}\t{item.status}\t{records}\t{item.generated_at}\t{item.summary_path}")
        lines.append(f"summaries: {len(summaries)}")
        return "\n".join(lines)

    paths = sorted((ROOT / "summaries" / "topics").glob("*.md"))
    if not paths:
        return "No topic summary files found."
    lines = ["topic\tstatus\trecords\tgenerated_at\tpath"]
    for path in paths:
        lines.append(f"{path.stem}\tfile_only\t\t\t{path}")
    lines.append(f"summaries: {len(paths)}")
    return "\n".join(lines)


## User Interface: ##

**Question** - answers questions using the pre-processed topic summaries from the conference list above. This requires an OpenAI key. Question types:
 
- Free form questions like "give me a complete report on vulnerability research" are answered from the 36 topic summaries compiled across every conference in the database.
- Conference questions like "tell me all about defcon 34" or another con in the data (see list below)

**Query**  - runs keyword searches for talks, tools, researchers, etc if you know what you're after - deterministic annotation/database SQL queries and does not call an LLM, no key required.
  
**Topics**  - lists the topic names; each topic has a topic summary created from the entire record set.

**Conferences in the data:**
  
- All years: CAMLIS, PROMPT|GTFO, Unprompted.
- 2025 conferences: DEF CON, Blackhat, BSides LV.
- 2026 conferences: DEF CON, Blackhat, BSides LV.

In [6]:
#@title Analyst Interface
# Query interface

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, Markdown
except ImportError:
    widgets = None
    print("ipywidgets is not installed. Use the helper functions directly:")
    print('  ask_summaries_only("...", max_summaries=3)')
    print('  query_annotations("prompt injection", limit=10)')
    print("  analyst_status()")
    print("  list_summary_topics()")
    print("  list_topic_summaries()")

if widgets is not None:
    mode = widgets.ToggleButtons(
        options=[ ("Query - SQL, no LLM", "query"), ("Question - calls LLM", "question"),("Topics", "topics"), ("Summary files", "summaries"), ("Status", "status")],
        value="query",
        description="Mode:",
        style={"description_width": "initial"},
    )
    text = widgets.Textarea(
        value="",
        placeholder="Ask a question of the knowledge agent or query the database records.",
        description="Input:",
        layout=widgets.Layout(width="100%", height="110px"),
        style={"description_width": "initial"},
    )
    limit = widgets.IntText(value=36, description="Query limit:", style={"description_width": "initial"})
    max_summaries = widgets.IntSlider(
        value=3,
        min=1,
        max=5,
        step=1,
        description="Max summaries:",
        style={"description_width": "initial"},
    )
    run_button = widgets.Button(description="Run", button_style="danger")
    status = widgets.HTML(
        value="<span style='color:#666'>Idle.</span>",
        layout=widgets.Layout(min_height="24px"),
    )
    display(HTML("""
    <style>
      .analyst-rendered-output,
      .analyst-rendered-output * {
        color: #1f2937 !important;
      }
      .analyst-rendered-output {
        font-family: var(--jp-content-font-family, system-ui, -apple-system, Segoe UI, sans-serif);
        line-height: 1.55;
        white-space: normal;
      }
      .analyst-rendered-output pre,
      .analyst-rendered-output code {
        background: #f3f4f6;
        color: #111827 !important;
      }
      .analyst-rendered-output-text {
        white-space: pre-wrap;
        margin: 0;
        font-family: var(--jp-code-font-family, Consolas, monospace);
        line-height: 1.45;
        color: #1f2937 !important;
      }
    </style>
    """))

    def set_status(message: str, color: str = "#666") -> None:
        status.value = f"<span style='color:{color}'>{message}</span>"

    def render_markdown_result(result: str) -> str:
        """Render Analyst output as Markdown-styled HTML with explicit text color."""
        safe_result = html.escape(result.replace("```", "'''"))
        try:
            import markdown as markdown_lib

            body = markdown_lib.markdown(safe_result, extensions=["extra", "nl2br"])
        except ImportError:
            body = "<p>" + safe_result.replace("\n", "<br>") + "</p>"
        return '<div class="analyst-rendered-output">' + body + "</div>"

    def linkify_plain_text(value: str) -> str:
        """Escape plain text and make URLs clickable."""
        url_re = re.compile(r"https?://[^\s<>()]+")
        parts: list[str] = []
        last = 0
        for match in url_re.finditer(value):
            parts.append(html.escape(value[last:match.start()]))
            url = match.group(0).rstrip(".,;:]")
            trailing = match.group(0)[len(url):]
            safe_url = html.escape(url, quote=True)
            parts.append(
                f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" '
                f'style="color:#0645ad !important; text-decoration:underline;">'
                f'{html.escape(url)}</a>'
            )
            parts.append(html.escape(trailing))
            last = match.end()
        parts.append(html.escape(value[last:]))
        return "".join(parts)

    def render_text_result(result: str) -> str:
        """Render structured database output with clickable URLs and dark text."""
        return '<div class="analyst-rendered-output-text">' + linkify_plain_text(result) + "</div>"

    def run_analyst(_button):
        run_button.disabled = True
        run_button.description = "Running..."
        set_status("Processing question...")
        result_display.update(HTML(""))

        try:
            render_as_text = False
            if mode.value == "status":
                set_status("Checking status...")
                result = analyst_status()
                render_as_text = True
            elif mode.value == "topics":
                set_status("Listing summary topics...")
                result = list_summary_topics()
                render_as_text = True
            elif mode.value == "summaries":
                set_status("Listing topic summary files...")
                result = list_topic_summaries()
                render_as_text = True
            elif mode.value == "query":
                set_status("Searching annotations/database...")
                result = query_annotations(text.value, limit=max(1, int(limit.value)))
                render_as_text = True
            else:
                set_status("NOTE: this agent is a cybersecurity research conference analyst and answers questions using the prepared conference topic summaries, not general knowledge. Hit Topics or Summary files for more information. ")
                result = ask_summaries_only(text.value, max_summaries=max_summaries.value)

            result_display.update(HTML(render_text_result(result) if render_as_text else render_markdown_result(result)))
            set_status("Done.")
        except Exception as exc:
            result_display.update(HTML(render_text_result(f"ERROR: {exc}")))
            set_status("Error. See output below.", "#b00020")
        finally:
            run_button.disabled = False
            run_button.description = "Run"

    run_button.on_click(run_analyst)
    display(widgets.VBox([mode, text, limit, max_summaries, run_button, status]))
    result_display = display(HTML(""), display_id=True)

In [7]:
#@title Examples
# Test / examples of how to make method calls::

# No-LLM status check.
# print(analyst_status())

# Deterministic, no-LLM annotation query.
# print(query_annotations("prompt injection", limit=5))

# List summary-ready topics and existing topic summary files.
# print(list_summary_topics())
# print(list_topic_summaries())

# Summaries-only LLM answer. Requires provider credentials in .env, Colab Secrets, or set_openai_key().
# print(ask_summaries_only("What do the summaries say about prompt injection?", max_summaries=3))